In [1]:
#Import necessary modules
import pandas as pd
import numpy as np
import ast
from scipy.stats import ranksums,wilcoxon
#kmeans
from sklearn.cluster import KMeans
from sklearn import preprocessing
from sklearn.metrics import silhouette_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import statistics
from collections import Counter

import math
from scipy.stats import variation
from scipy.stats import iqr
from scipy import stats 

import random
from random import seed
from random import randint
from sklearn.neighbors import LocalOutlierFactor
from pyod.models.lof import LOF
from pyod.models.ocsvm import OCSVM
from sklearn.metrics import confusion_matrix
import scipy.stats
import scipy.stats as st
from scipy.stats import t
from collections import defaultdict
import pandas as pd
from sklearn.ensemble import IsolationForest
from statistics import median
# from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
patients=[1135, 1450, 1464, 1497, 1504, 
          1511, 1541, 1559, 1572, 1582, 
          1586, 1603, 1607, 1609, 1615, 
          1617, 1628, 1657, 1660, 1706]

In [3]:
def custom_sort_key(item):
    from datetime import datetime
    # Parse the date and time using strptime and convert to a sortable format
    return datetime.strptime(item[0], '%d/%m/%Y %H:%M:%S'), item[1]


In [4]:
def read_left(patient):
    threshold=1800
    try:
        #file.csv is an empty csv file
        tm = pd.read_csv(f"{patient,threshold}_left_days_outing_living"'.csv')
        # Convert 'Date' to datetime format
        tm['Date'] = pd.to_datetime(tm['Date'])
        tm['Date'] =tm['Date'].dt.strftime('%d/%m/%Y')
        left_days_hours = np.array(tm).tolist() 
    
    except pd.errors.EmptyDataError:
        left_days_hours =[]
    return left_days_hours

In [5]:
def readData(patient):
    tm = pd.read_csv(f"{patient}_cleaned_removedays"'.csv') #solved double coverage problem
    out = np.array(tm).tolist()  

    for i in range(len(out)):
        if 'Bedroom' in out[i][1]:
            out[i][1]='Bedroom'
            
        if 'Bathroom' in out[i][1]:
            out[i][1]='Bathroom'
    
        if 'Sensor Line' in out[i][1] or 'Extra sensor line' in out[i][1]:
            out[i][1]='Hallway'
            
        if 'Hallway' in out[i][1]:
            out[i][1]= 'Hallway'
            
        if 'Walk-in Closet' in out[i][1]:
            out[i][1]= 'Walk-in Closet'
            
        if 'Kitchen' in out[i][1]:
            out[i][1]= 'Kitchen'
            
        if 'Dining Room' in out[i][1]:
            out[i][1]= 'Dining Room'
            
#         if 'Lounge' in out[i][1]:
#             out[i][1]= 'Lounge'
    
    
    return out

In [6]:
# Function to replace specific room names with a generalized category
def replace_room_name(room):
    if 'Bathroom' in room:
        return 'Bathroom'
    
    elif 'Bedroom' in room :
        return 'Bedroom'
    
    elif 'Kitchen' in room :
        return 'Kitchen'
    
    elif 'Dining Room' in room :
        return 'Dining Room'
    
    elif 'Living Room' in room :
        return 'Living Room'
    elif 'Sensor Line' in room or 'Extra sensor line' in room:
        return 'Hallway'
    elif 'Hallway' in room:
        return 'Hallway'
    else:
        return room


In [7]:
def generate_house_map(timestamp_data):
    house_map = {'rooms': set(), 'connections': set()}

    # Extract unique rooms from timestamp data
    for timestamp_event in timestamp_data:
        room_names = [room.strip() for room in timestamp_event[1].split(',')]
        house_map['rooms'].update(room_names)

    # Generate all possible connections between rooms
    all_rooms = list(house_map['rooms'])
    for room1 in all_rooms:
        for room2 in all_rooms:
            if room1 != room2:
                connection = (room1, room2)
                reverse_connection= (room2, room1)
                if connection not in house_map['connections'] and reverse_connection not in house_map['connections']:
                    house_map['connections'].add(connection)

    return {'rooms': list(house_map['rooms']), 'connections': list(house_map['connections'])}

In [8]:
def count_consecutive_activations(timestamp_data, house_map):
    connections_count = {connection: 0 for connection in house_map['connections']}

    active_rooms = set()

    for event in timestamp_data:
        timestamp, room = event
        active_rooms.add(room)

        # Check for consecutive activation between rooms
        for connection in house_map['connections']:
            room1, room2 = connection
            reverse_connection = (room2, room1)

            if room1 in active_rooms and room2 == room:
                connections_count[connection] += 1


            elif room2 in active_rooms and room1 == room:
                connections_count[connection] += 1

    # Sort connections by count in descending order
    sorted_connections = sorted(connections_count.items(), key=lambda x: x[1], reverse=True)
    return sorted_connections

In [9]:
def read_adjacent(patient):
    tm = pd.read_csv(f"{patient}_adjacent_list"'.csv').dropna() #drop 'NaN'
    adjacent = np.array(tm).tolist()  
   
    # Apply replacement and convert to bidirectional tuples
    bidirectional_adjacency = []
    for pair in adjacent:
        # Replace room names in each pair
        updated_pair = [replace_room_name(room) for room in pair]

        # Add both directional pairs
        bidirectional_adjacency.append(tuple(updated_pair))
        bidirectional_adjacency.append(tuple(updated_pair[::-1]))
    bidirectional_adjacency=list(set(bidirectional_adjacency))
    return bidirectional_adjacency

## 2. Entryway sensor firing time segment
### Double_coverages

In [10]:
def consecutive_firings(out):
    # Example usage
    timestamp_data =out
    house_map = generate_house_map(timestamp_data)
    sorted_connections = count_consecutive_activations(timestamp_data, house_map)
    # Separate connections with counts greater than zero and counts equal to zero
    non_zero_connections = []
    zero_connections = []
    for connection, count in sorted_connections:
        if count > 0:
            non_zero_connections.append(connection)
        else:
            zero_connections.append(connection)

    return non_zero_connections

In [11]:
# exit_rooms = ['Hallway','Front Door','Back Door','Other Door',
#               'Balcony Door','Other Door 2','Garage Door','Car 1','Balcony 1']#'Bathroom'

In [12]:
exit_rooms = ['Front Door','Back Door','Other Door',
              'Balcony Door','Other Door 2','Garage Door','Balcony 1']#'Bathroom'

In [13]:
def bidirectional(non_adjacent_area):
    bidirectional_non_adjacency=[]
    for pair in non_adjacent_area:
        # Replace room names in each pair
        updated_pair = [replace_room_name(room) for room in pair]
        # Add both directional pairs
        bidirectional_non_adjacency.append(tuple(updated_pair))
        bidirectional_non_adjacency.append(tuple(updated_pair[::-1]))
    bidirectional_non_adjacency=list(set(bidirectional_non_adjacency))
    return bidirectional_non_adjacency

In [14]:
# Helper function to strip numerical suffixes and get the base room name
def base_room_name(room):
    for criteria in criteria_categories:
        if room.startswith(criteria):
            return criteria
    return room

In [15]:
criteria_categories = ['Living Room', 'Kitchen', 'Dining Room']  
def visible_room(non_adjacent_area):
    visible=[]
    for pair in non_adjacent_area:
        for room in pair:
            if 'Hallway' in room:
                visible.append(pair)
    # Criteria for being a visible room: at least one from each category
    from itertools import combinations
    
    all_combinations = bidirectional(list(combinations(criteria_categories, 2)))

    for (room1,room2) in non_adjacent_area:       
        room1_base = base_room_name(room1)
        room2_base = base_room_name(room2)
        if (room1_base,room2_base) in all_combinations:
            visible.append((room1,room2))
        
    visible_rooms=bidirectional(visible)
    return visible_rooms 

In [16]:
def filtered_data_nonadjacent(possible_visitor_adjacent):
    result = []
    # Initialize a set to store the values in previous items
    previous_values = set()

    for item in possible_visitor_adjacent:
        first_instance, second_instance, value1, value2 = item

        # Check if any part of the values pair is not in previous items
        if (value1 not in previous_values) and (value2 not in previous_values):
            result.append(item)

        # Update the set of values in previous items
        previous_values.update([value1, value2])
    return result

In [17]:
def other_rooms(out):
    import datetime
   
    possible_visitor_non_adjacent=[]
    possible_regions_non_adjacent=[]
    possible_visitor_adjacent=[]
    possible_regions_adjacent=[]
    
    for i in range(len(out)):
        # find all active events
        area1 = out[i][1]

        for j in range(i+1,len(out)):
            #  if next signal is active and not sending from the same sensor 
            if  (out[i][1]!=out[j][1] ):
                area2=out[j][1]


                if (area1,area2) in non_adjacent_area:

#                     print('non_adjacent_visitor',area1,area2,out[i],out[j])
#                     possible_visitor_non_adjacent.append((out[i],out[j],i,j))
#                     possible_regions_non_adjacent.append((area1,area2,i,j))

#                 elif (area1,area2) not in non_adjacent_area and (area1,area2) not in visible_rooms:  
#                     #adjacent but not cause double coverage problem (NOT IN VISIBLE but adjacent)
#                     #check time difference
                    active_time1 = datetime.datetime.strptime(out[i][0], "%d/%m/%Y %H:%M:%S")
                    active_time2 = datetime.datetime.strptime(out[j][0], "%d/%m/%Y %H:%M:%S")

# #                      # find which area sends inactive signal first
# #                     index_inactive1=next(a for a in range(j,len(out)) if out[a][2]=='inactive' and out[a][1]==area1)
# #                     index_inactive2=next(b for b in range(j,len(out)) if out[b][2]=='inactive' and out[b][1]==area2)
# #                     inactive_time1 = datetime.datetime.strptime(out[index_inactive1][0], "%d/%m/%Y %H:%M:%S")
# #                     inactive_time2 = datetime.datetime.strptime(out[index_inactive2][0], "%d/%m/%Y %H:%M:%S")

# #                     diff_i = (inactive_time1-active_time2).seconds
                    diff_a = (active_time2-active_time1).seconds
# #                     if diff_i>0: #second region activate the same time
# #                         print('abnormal-visitor activate at same time',area1,area2,i)
# #                         possible_visitor_adjacent.append((out[i],out[j],i,j))
# #                         possible_regions_adjacent.append((area1,area2,i,j))

                    if diff_a < 1:
                        possible_visitor_non_adjacent.append((out[i],out[j],i,j))
#                     possible_regions_non_adjacent.append((area1,area2,i,j))
        
        
#                         print('non_adjacent_visitor',area1,area2, active_time1,active_time2)
#                         print('abnormal-visitor activate at same time',area1,area2, active_time1,active_time2)
#                         print(out[i],out[j],out[index_inactive1],diff_i)
#                         possible_visitor_adjacent.append((out[i],out[j],i,j))
#                         possible_regions_adjacent.append((area1,area2,i,j))
            break
#     print('original',possible_visitor_adjacent)
    possible_visitor_non_adjacent=filtered_data_nonadjacent(possible_visitor_non_adjacent)
#     possible_visitor_adjacent=filtered_data_adjacent(possible_visitor_adjacent)
    
    return possible_visitor_non_adjacent        






####  If two non-adjacent sensors activate at the same time, there is a need to identify which is the real region a user is positioned.

1. If second sensor activated before first one is inactive: two sensors activated at the same time
    
    if two sensors are in non-adjacent region:

    •	If it only triggered once/only in 1 unreasonable region---it may be a sensor problem.
    
    •	If sensors triggered in many unreasonable regions, then it more likely to be triggered by other guests. 
        ---trace when the weird data ends.


####  Additional people

When second people enters into the house, it is hard to identify which motion is caused by the resident. To maitain the accuracy of later calculation, it is better to identify the "coming in" and "leaving" time by the visitors, so that we can disgard all records during the time period.


In [18]:
def readdate(out): #read all date
    new_date=[]
    for i in range(len(out)):
        dt = out[i][0].split(' ')[0]
        if dt not in new_date:
            new_date.append(dt)
    return new_date

### Non-adjacent --infer visitor

In [19]:
def non_adjacent_hour(possible_visitor):                 
    from datetime import datetime  
    tm=[]
    for item in possible_visitor:
        room1=item[0][1]
        room2=item[1][1]
        t0=item[0][0].split(' ')
        t1=item[1][0].split(' ')
        start=t0[1]
        end =t1[1]
        import datetime
        start_dt = datetime.datetime.strptime(start, '%H:%M:%S')
        end_dt = datetime.datetime.strptime(end, '%H:%M:%S')
#     #     diff=(end_dt-start_dt).seconds
#         new_record.append([t0[0],[t0[1],t1[1]]])

        #(1) inactivity period within an hour
        if start_dt.hour == end_dt.hour:
            #  calculate inactive time
#             diff=(end_dt-start_dt).seconds
            h = (str(start_dt.hour))
            tm.append([t0[0],[room1,room2],h])

        #(2) if within same day inactive over an hour boundary
        if end_dt.hour!=start_dt.hour:
            if end_dt.hour !=0 :
#                 diff = (end_dt-start_dt).seconds
                h = (str(start_dt.hour))
                tm.append([t0[0],[room1,room2],h])

                for i in range(1,end_dt.hour-start_dt.hour):
                    hm = start_dt.hour+i
                    h2 = (str(hm))
                    tm.append([t0[0],[room1,room2],h2])

                h2 = (str(end_dt.hour))
                tm.append([t0[0],[room1,room2],h2])


        #(3) overlap hours to next day 

        if end_dt.hour!=start_dt.hour:
            if end_dt.hour == 0:
#                 diff = (end_dt-start_dt).seconds
                dt = datetime.datetime.strptime(t0[0], "%d/%m/%Y")
                one_day = datetime.timedelta(days=1)
                new_dt=(dt+one_day).strftime("%d/%m/%Y")                    
                #change date duplicate records 
                h1 = (str(start_dt.hour))
                tm.append([t0[0],[room1,room2],h1])

                for i in range(start_dt.hour+1,24):
                    hm = i
                    h2 = (str(hm))
                    tm.append([t0[0],[room1,room2],h2])
#                     print(i)
                h2 = (str(end_dt.hour))
                tm.append([new_dt, [room1,room2],h2])
    return tm               

In [20]:
 # if left more than one time within the same hour, added time together
def visitor_days(possible_days):
    new=[]
    new2=[]
    for each in possible_days:
        new.append([each[0],each[2]])
    
    #remove duplicate (keep the order)
    for each in new:
        if each not in new2:
            new2.append(each)
            
    return new2

In [21]:
def visitor_days_twice(data):
    # Dictionary to store room combinations for each date and hour
    room_combinations = defaultdict(set)

    # List to store the result
    result = []

    # Iterate through the data
    for entry in data:
        date, rooms, hour = entry[0], entry[1], entry[2]
        room_combination_key = (date, hour)
        room_combinations[room_combination_key].add(tuple(sorted(rooms)))

    # Check for at least two different room combinations at the same date and hour
    for key, value in room_combinations.items():
        if len(value) >= 2:
            result.append([key[0], key[1]])
            
    return result

In [22]:
from datetime import datetime

def convert_firings_numbers(sensor_data):
    # Convert the timestamp strings to datetime objects and sort the data
    sensor_data = sorted(
        [(datetime.strptime(event[0], "%d/%m/%Y %H:%M:%S"), event[1]) for event in sensor_data],
        key=lambda x: x[0]
    )

    # Encode locations to odd numbers
    locations = sorted(set(event[1] for event in sensor_data))
    location_to_number = {loc: idx * 2 + 1 for idx, loc in enumerate(locations)}
    print(location_to_number)
    
    
    # Prepare the structure for the AI vectors by day
    ai_vectors_by_day = {}

    for timestamp, location in sensor_data:
        date_key = timestamp.strftime('%d/%m/%Y')
        hour = timestamp.hour
        minute = timestamp.minute

        # Initialize the day and hour structures if they don't exist
        if date_key not in ai_vectors_by_day:
            ai_vectors_by_day[date_key] = [[None for _ in range(60)] for _ in range(24)]
        day_vector = ai_vectors_by_day[date_key]

        # Assign the location number based on the first active status in a minute
        if day_vector[hour][minute] is None:
            location_code = location_to_number[location]
            day_vector[hour][minute] = location_code

    # Fill forward the last known location within each hour and across hours
    for day, hours in ai_vectors_by_day.items():
        for h in range(24):
            for m in range(60):
                # Fill in the gaps within the hour
                if hours[h][m] is None:
                    hours[h][m] = hours[h][m-1] if m > 0 else (hours[h-1][-1] if h > 0 else 0)
    
    return ai_vectors_by_day

In [23]:
import csv

for num in patients:
    print('patient',num)
    patient = num
   
    out=readData(patient)
    adjacent=read_adjacent(patient)

    new_date=readdate(out)

    remove=[]
    for i in range(len(out)):
        dt = out[i][0].split(' ')[0]
        if dt == new_date[-1]:
            remove.append(out[i])
    out=[a for a in out if a not in remove]
    #remove DATE
    new_date=new_date[:-1]
    
    
         
#     #(1) mapping locations to odd numbers
    ai_vectors_by_day=convert_firings_numbers(out)
    
    # Open a new CSV file in write mode
    with open(f'Patient{num}_mapping_entropy.csv', 'w', newline='') as file:

        # Create a CSV writer object
        writer = csv.writer(file)
        # Write the header row
        writer.writerow(ai_vectors_by_day.keys())
        # Write data rows
        writer.writerows(zip(*ai_vectors_by_day.values()))
        
    df_entropy =  pd.DataFrame (out)
    df_entropy.to_csv(f"{patient}_cleaned_Step2_entropy_alldata"'.csv', index=False, header=True)

     #(2)  find time period between 'Entryway Motion'  (from active1 to inactive 2)
    
    consecutive_firing =consecutive_firings(out)
    
    #non_adjacent_area: those in consecutive_firing(out) but not in adjacent

    non_adjacent_area = [a for a in consecutive_firing if a not in adjacent]

    visible_rooms=visible_room(non_adjacent_area)
    non_adjacent_area=[a for a in bidirectional(non_adjacent_area) if a not in visible_rooms]
  
    
     
    #(2.1) double coverage problem
 
        
    #(2.2+2.3) adjacent & non_adjacent_area problem                
    possible_visitor_non_adjacent=other_rooms(out)

    

#     # (3) save to csv
    df =  pd.DataFrame (out)
    df.to_csv(f"{patient}_cleaned_Step2_alldata"'.csv', index=False, header=True)
    print("saved file")
    
    #visitor_days_once   
#     # possible_visitor--non_adjacent
#     possible_visitor_non_adjacent_final=visitor_days(non_adjacent_hour(possible_visitor_non_adjacent))
#     df2 =  pd.DataFrame (possible_visitor_non_adjacent_final)
#     df2.to_csv(f"{patient}_possible_visitor_non_adjacent"'.csv', index=False, header=True)
    
#visitor_days_twice
    possible_visitor_non_adjacent_final_twice=visitor_days_twice(non_adjacent_hour(possible_visitor_non_adjacent))
    df2_twice =  pd.DataFrame (possible_visitor_non_adjacent_final_twice)
    df2_twice.to_csv(f"{patient}_possible_visitor_non_adjacent_twice_alldata"'.csv', index=False, header=True)
    print("saved non_adjacent")
    
    
# # #     #visitor_days_once--adjacent
# #     possible_visitor_adjacent_final=visitor_days(non_adjacent_hour(possible_visitor_adjacent))
# #     df3 =  pd.DataFrame (possible_visitor_adjacent_final)
# #     df3.to_csv(f"{patient}_possible_visitor_visitor_adjacent"'.csv', index=False, header=True)
    
# # # #visitor_days_twice
#     possible_visitor_adjacent_final_twice=visitor_days_twice(non_adjacent_hour(possible_visitor_adjacent))
#     df3_twice =  pd.DataFrame (possible_visitor_adjacent_final_twice)
#     df3_twice.to_csv(f"{patient}_possible_visitor_visitor_adjacent_twice_alldata"'.csv', index=False, header=True)



patient 1135
{'Back Door': 1, 'Bathroom': 3, 'Bedroom': 5, 'Dining Room': 7, 'Front Door': 9, 'Hallway': 11, 'Kitchen': 13, 'Living Room': 15, 'Other 1': 17}
saved file
saved non_adjacent
patient 1450
{'Bathroom': 1, 'Bedroom': 3, 'Front Door': 5, 'Hallway': 7, 'Kitchen': 9, 'Living Room': 11, 'Walk-in Closet': 13}
saved file
saved non_adjacent
patient 1464
{'Back Door': 1, 'Bathroom': 3, 'Bedroom': 5, 'Front Door': 7, 'Hallway': 9, 'Kitchen': 11, 'Living Room': 13, 'Other 1': 15}
saved file
saved non_adjacent
patient 1497
{'Bathroom': 1, 'Bedroom': 3, 'Front Door': 5, 'Hallway': 7, 'Kitchen': 9, 'Living Room': 11}
saved file
saved non_adjacent
patient 1504
{'Back Door': 1, 'Bathroom': 3, 'Bedroom': 5, 'Front Door': 7, 'Hallway': 9, 'Kitchen': 11, 'Laundry Room 1': 13, 'Living Room': 15, 'Lounge 1': 17, 'Office 1': 19, 'Other 1': 21}
saved file
saved non_adjacent
patient 1506
{'Balcony Door': 1, 'Bathroom': 3, 'Bedroom': 5, 'Front Door': 7, 'Hallway': 9, 'Kitchen': 11, 'Living Room': 1

In [24]:
read_adjacent(1497)

[('Kitchen', 'Living Room'),
 ('Bathroom', 'Hallway'),
 ('Hallway', 'Bathroom'),
 ('Living Room', 'Kitchen'),
 ('Bedroom', 'Hallway'),
 ('Living Room', 'Hallway'),
 ('Living Room', 'Front Door'),
 ('Hallway', 'Bedroom'),
 ('Front Door', 'Hallway'),
 ('Front Door', 'Living Room'),
 ('Hallway', 'Living Room'),
 ('Hallway', 'Front Door')]

In [25]:
# def wrong(out):
#     import datetime
#     wrong_region_records=[]
#     possible_visitor_non_adjacent=[]
#     possible_regions_non_adjacent=[]
#     possible_visitor_adjacent=[]
#     possible_regions_adjacent=[]
    
#     for i in range(len(out)):
#         # find all active events
#         if out[i][2]=='active':
#             area1 = out[i][1]
  
#             for j in range(i+1,len(out)):
#                 #  if next signal is active and not sending from the same sensor 
#                 if  (out[i][1]!=out[j][1] ) and  (out[j][2]=='active'):
#                     area2=out[j][1]
                    
#                     # if two area are in visible_area
#                     if (area1,area2) in visible_area:  #adjacent problem
# #                         print('visible_area',area1,area2,i,j)    
#                         # find which area sends inactive signal first
#                         index_inactive1=next(a for a in range(j,len(out)) if out[a][2]=='inactive' and out[a][1]==area1)
#                         index_inactive2=next(b for b in range(j,len(out)) if out[b][2]=='inactive' and out[b][1]==area2)

#                         # the "wrong" region sends normal signal while "correct" region sends violated signal
#                         # the "wrong" region will send inactive signal before "correct" region

#                         # find inactive signal in both regions
#                         # identify which region sends ealier inactive signal
#                         #---"wrong" region and append both active & inactive events to a list

#                         inactive_time1 = datetime.datetime.strptime(out[index_inactive1][0], "%d/%m/%Y %H:%M:%S")
#                         inactive_time2 = datetime.datetime.strptime(out[index_inactive2][0], "%d/%m/%Y %H:%M:%S")

#                         if inactive_time1>=inactive_time2: # second region ceased earlier than first region

#                             #Ka-La-Li-Ki (lounge is a wrong region)

#                             # double coverage issue, area2 is a "wrong" region
#                             wrong_region=area2
#                             wrong_region_records.append(out[j])
#                             wrong_region_records.append(out[index_inactive2])
                            
#                     elif (area1,area2) in non_adjacent_area:
                        
#                         print('non_adjacent_visitor',area1,area2,i,j)
#                         possible_visitor_non_adjacent.append((out[i],out[j],i,j))
#                         possible_regions_non_adjacent.append((area1,area2,i,j))
                        
#                     elif ((area1,area2) not in visible_area) and ((area1,area2) not in non_adjacent_area):  
#                         #adjacent but not cause double coverage problem (NOT IN VISIBLE but adjacent)
#                         #check time difference
#                         active_time1 = datetime.datetime.strptime(out[i][0], "%d/%m/%Y %H:%M:%S")
#                         active_time2 = datetime.datetime.strptime(out[j][0], "%d/%m/%Y %H:%M:%S")
            
#                          # find which area sends inactive signal first
#                         index_inactive1=next(a for a in range(j,len(out)) if out[a][2]=='inactive' and out[a][1]==area1)
#                         index_inactive2=next(b for b in range(j,len(out)) if out[b][2]=='inactive' and out[b][1]==area2)
#                         inactive_time1 = datetime.datetime.strptime(out[index_inactive1][0], "%d/%m/%Y %H:%M:%S")
#                         inactive_time2 = datetime.datetime.strptime(out[index_inactive2][0], "%d/%m/%Y %H:%M:%S")

#                         diff_i = (inactive_time1-active_time2).seconds
#                         diff_a = (active_time2-active_time1).seconds
#                         if diff_i>0: #second region activate the same time
#                             print('abnormal-visitor activate at same time',area1,area2,i)
#                             possible_visitor_adjacent.append((out[i],out[j],i,j))
#                             possible_regions_adjacent.append((area1,area2,i,j))
                      
# #                         elif diff_a=<1:
# #                             print('abnormal-visitor diff_a=<1',area1,area2)
# #                             print(out[i],out[j],out[index_inactive1],diff_i)
# #                             possible_visitor_adjacent.append((out[i],out[j],i,j))
# #                             possible_regions_adjacent.append((area1,area2,i,j))
#                 break
#     print('original',possible_visitor_adjacent)
#     possible_visitor_non_adjacent=filtered_data_nonadjacent(possible_visitor_non_adjacent)
# #     print('non_adjacent_visitor',possible_visitor_non_adjacent)
    
#     possible_visitor_adjacent=filtered_data_adjacent(possible_visitor_adjacent)
    
#     return wrong_region_records,possible_visitor_non_adjacent,possible_visitor_adjacent        


In [26]:
xxx

NameError: name 'xxx' is not defined

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Your provided data
data = out

# Create a directed graph
G = nx.DiGraph()

# Add nodes and edges based on transitions
for i in range(1, len(data)):
    timestamp_prev, area_prev, state_prev = data[i - 1]
    timestamp_curr, area_curr, state_curr = data[i]

    # Add nodes for current and previous areas
    G.add_node(area_prev)
    G.add_node(area_curr)

    # Add an edge if there is a transition from active to inactive
    if state_prev == 'active' and state_curr == 'inactive':
        G.add_edge(area_prev, area_curr)

# Draw the graph
pos = nx.spring_layout(G)  # You can use other layout algorithms
nx.draw(G, pos, with_labels=True, font_weight='bold', node_size=700, node_color='skyblue', font_color='black', font_size=8)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_room_connection_counts(connections_count):
    connections, counts = zip(*connections_count.items())

    plt.figure(figsize=(20, 6))
    plt.bar(range(len(connections)), counts, color='skyblue')
    plt.xlabel('Room Connections')
    plt.ylabel('Count of Consecutive Activations')
    plt.title('Consecutive Activations Between Room Connections')
    plt.xticks(range(len(connections)), connections, rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Example usage
timestamp_data = out
house_map = generate_house_map(timestamp_data)
connections_count = count_consecutive_activations(timestamp_data, house_map)
plot_room_connection_counts(dict(connections_count))


In [ ]:
# def convert_firings_numbers(out):
#     locations = [d[1] for d in out]
#     odd_numbers = (range(1, len(set(locations))+1 )) # generate a list of odd numbers
#     location_to_number = dict(zip(set(locations), odd_numbers)) # create a dictionary mapping locations to odd numbers
    

#     from datetime import datetime, timedelta

#     #  mapping  room names to numbers
#     room_map =location_to_number

#     # Define the time range to loop over
#     start_date = datetime.strptime((new_date[0]),"%d/%m/%Y")
#     end_date = datetime.strptime((new_date[-1]),"%d/%m/%Y")

#     # Loop over each day and calculate the ai_vectors
#     ai_vectors_by_day = {}

#     for i in range((end_date - start_date).days + 1):
#         current_date = start_date + timedelta(days=i)

#         # Find the first active room in the data
#         first_active_room = location_to_number[out[0][1]]
#         # Initialize prev_location to be the first active room
#         prev_location = first_active_room


#         ai_vectors = []
#         for hour in range(24):
#             # Define the start and end times for the hour
#             start_time = f'{current_date.strftime("%d/%m/%Y")} {hour:02d}:00:00'
#             end_time = f'{current_date.strftime("%d/%m/%Y")} {hour:02d}:59:59'

#             hour_locations = []

#             for minute in range(60):
#                  # Find the first active location in this minute
#                 active_location = None
#                 current_time = f'{current_date.strftime("%d/%m/%Y")} {hour:02d}:{minute:02d}:00'
#                 for i in range(len(out)):
#                 # Find the first activated sensor in the current minute, if any
#                     if start_time <= out[i][0] <= end_time and current_time <= out[i][0] < f'{current_date.strftime("%d/%m/%Y")} {hour:02d}:{minute+1:02d}:00':
#     #                     # Only consider the first activated location

#                         active_location =  location_to_number[out[i][1]]
#                         prev_location = active_location   
#     #                     print(data[i][0],prev_location)
#                         break

#                          # If there is no active location in this minute, use the previous location
#                     # If there is no active location in this minute, use the previous location
#                 if active_location is None:
#                     hour_locations.append(prev_location)
#                 else:
#                     hour_locations.append(prev_location)

#     #                 hour_locations.append(prev_location)


#     #             print(current_time,active_location )
#     #             hour_locations.append(prev_location)
#             ai_vectors.append(hour_locations)
#         ai_vectors_by_day[current_date.strftime('%d/%m/%Y')] = ai_vectors
#     return ai_vectors_by_day